# 01 - Basic RAG Pipeline

এই নোটবুকে আমরা একটি সহজ RAG pipeline বানাবো।

**Topics:**
- Document corpus তৈরি
- TF-IDF vectorization
- Cosine similarity দিয়ে retrieval
- N-gram generation
- Full RAG pipeline
- Evaluation


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Document Corpus তৈরি


In [ ]:
documents = [
    "Machine learning is a subset of artificial intelligence that enables computers to learn from data.",
    "Deep learning uses neural networks with many layers to model complex patterns in data.",
    "Natural language processing allows computers to understand and generate human language.",
    "Computer vision enables machines to interpret and understand visual information from images.",
    "Reinforcement learning is a type of machine learning where agents learn by interacting with an environment.",
    "Supervised learning uses labeled data to train models for prediction tasks.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning allows a model trained on one task to be reused on a related task.",
    "Generative AI can create new content such as text, images, and music.",
    "Large language models are trained on vast amounts of text data to understand and generate language."
]

print(f'Total documents: {len(documents)}')
for i, doc in enumerate(documents, 1):
    print(f'  {i}. {doc}')


## 2. Text Preprocessing


In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

sample = documents[0]
print(f'Original: {sample}')
print(f'Tokens: {preprocess(sample)}')


## 3. TF-IDF from Scratch


In [ ]:
class TFIDF:
    def __init__(self):
        self.idf = {}
        self.vocab = []
        self.vocab_index = {}
    
    def fit(self, docs):
        tokenized = [preprocess(d) for d in docs]
        all_tokens = set()
        for tokens in tokenized:
            all_tokens.update(tokens)
        self.vocab = sorted(list(all_tokens))
        self.vocab_index = {w: i for i, w in enumerate(self.vocab)}
        N = len(docs)
        for word in self.vocab:
            df = sum(1 for tokens in tokenized if word in tokens)
            self.idf[word] = math.log(N / (df + 1)) + 1
        return self
    
    def transform(self, docs):
        vectors = []
        for doc in docs:
            tokens = preprocess(doc)
            tf = Counter(tokens)
            vec = np.zeros(len(self.vocab))
            for word, count in tf.items():
                if word in self.vocab_index:
                    idx = self.vocab_index[word]
                    vec[idx] = count * self.idf[word]
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec = vec / norm
            vectors.append(vec)
        return np.array(vectors)
    
    def fit_transform(self, docs):
        return self.fit(docs).transform(docs)

tfidf = TFIDF()
doc_vectors = tfidf.fit_transform(documents)
print(f'TF-IDF shape: {doc_vectors.shape}')
print(f'Vocab size: {len(tfidf.vocab)}')


## 4. Cosine Similarity


In [ ]:
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

sim = cosine_similarity(doc_vectors[0], doc_vectors[1])
print(f'Cosine similarity (doc0 vs doc1): {sim:.4f}')

sim_matrix = np.zeros((len(documents), len(documents)))
for i in range(len(documents)):
    for j in range(len(documents)):
        sim_matrix[i, j] = cosine_similarity(doc_vectors[i], doc_vectors[j])

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(len(documents)), [f'D{i+1}' for i in range(len(documents))])
plt.yticks(range(len(documents)), [f'D{i+1}' for i in range(len(documents))])
plt.title('Document Similarity Matrix')
plt.tight_layout()
plt.show()


## 5. N-gram Generation


In [ ]:
def generate_ngrams(tokens, n=2):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

tokens = preprocess(documents[0])
bigrams = generate_ngrams(tokens, 2)
trigrams = generate_ngrams(tokens, 3)
print(f'Tokens: {tokens}')
print(f'Bigrams: {bigrams}')
print(f'Trigrams: {trigrams}')

all_bigrams = []
for doc in documents:
    all_bigrams.extend(generate_ngrams(preprocess(doc), 2))
bigram_freq = Counter(all_bigrams)
top_bigrams = bigram_freq.most_common(10)
plt.figure(figsize=(10, 5))
plt.barh([b[0] for b in top_bigrams], [b[1] for b in top_bigrams])
plt.xlabel('Frequency')
plt.title('Top 10 Bigrams')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 6. Full RAG Pipeline


In [ ]:
class BasicRAG:
    def __init__(self, documents):
        self.documents = documents
        self.tfidf = TFIDF()
        self.doc_vectors = self.tfidf.fit_transform(documents)
    
    def retrieve(self, query, top_k=3):
        query_vec = self.tfidf.transform([query])[0]
        similarities = [(i, cosine_similarity(query_vec, doc_vec)) for i, doc_vec in enumerate(self.doc_vectors)]
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:top_k]
    
    def generate(self, query, top_k=3):
        retrieved = self.retrieve(query, top_k)
        context = ' '.join([self.documents[i] for i, _ in retrieved])
        return {'query': query, 'retrieved': [(i, self.documents[i], s) for i, s in retrieved], 'context': context, 'response': f'Based on: {context}'}

rag = BasicRAG(documents)
print('RAG system initialized!')


In [ ]:
queries = ['What is machine learning?', 'How do neural networks work?', 'Tell me about language models']
for query in queries:
    print(f'Query: {query}')
    result = rag.generate(query, top_k=2)
    for i, doc, score in result['retrieved']:
        print(f'  Doc {i+1} (score: {score:.4f}): {doc}')
    print()


## 7. Evaluation


In [ ]:
def precision_at_k(relevant, k):
    return sum(relevant[:k]) / k

def recall_at_k(relevant, total):
    return sum(relevant) / total

query = 'machine learning algorithms'
retrieved = rag.retrieve(query, top_k=5)
relevant_docs = {0, 4, 5}
retrieved_indices = [i for i, _ in retrieved]
retrieved_relevant = [1 if i in relevant_docs else 0 for i in retrieved_indices]
print(f'Query: {query}')
print(f'Precision@3: {precision_at_k(retrieved_relevant, 3):.4f}')
print(f'Recall@3: {recall_at_k(retrieved_relevant, 3):.4f}')

scores = [score for _, score in retrieved]
labels = [f'D{i+1}' for i, _ in retrieved]
colors = ['green' if i in relevant_docs else 'red' for i, _ in retrieved]
plt.figure(figsize=(8, 4))
plt.bar(labels, scores, color=colors)
plt.ylabel('Similarity Score')
plt.title('Retrieval Results')
plt.tight_layout()
plt.show()


## 8. Exercises


In [ ]:
# Exercise 1: Implement BM25 scoring
# Exercise 2: Add query expansion
# Exercise 3: Implement reranking
# Exercise 4: Add more documents
print('Exercises listed above!')
